# Funding-Rate Carry with a Scalar Kalman Filter
## An annotated, step-by-step notebook

This notebook decomposes the program into individual components:

1. Load perpetual funding-rate data for a configurable exchange and symbol.
2. Understand the carry trade: long spot + short perpetual.
3. Model noisy funding with a one-dimensional Kalman filter.
4. Inspect one Kalman update by hand.
5. Filter the full series and visualise smoothing behaviour.
6. Convert the filtered estimate into a cost-aware position signal.
7. Backtest without look-ahead bias.
8. Tune the smoothing parameter on a training period only.
9. Evaluate on a held-out test period.
10. Examine limitations and possible improvements.

The notebook mirrors the logic of `funding_carry.py`, but exposes intermediate objects such as the innovation, Kalman gain, predicted variance, posterior variance, positions, turnover, and per-bar P&L.

## 0. Strategy in one picture

The trade is intended to be approximately delta neutral:

$$
\mathrm{position} = +1\,\mathrm{coin}_{\mathrm{spot}} - 1\,\mathrm{coin}_{\mathrm{perp}}
$$

If the funding rate is positive, the short perpetual leg receives funding. In this simplified research model:

$$
\mathrm{PnL}^{\mathrm{gross}}_t =
\begin{cases}
f_t N, & \mathrm{open}\\
0, & \mathrm{flat}
\end{cases}
$$

where $f_t$ is the realised relative funding rate for the selected market and $N$ is notional.

The Kalman filter does **not** create the arbitrage. It estimates whether the funding level is sufficiently persistent and positive to justify paying trading costs.

In [ ]:
# Imports and configuration
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:.8e}")


@dataclass(frozen=True)
class MarketConfig:
    exchange: str
    symbol: str
    cache_dir: Path = Path("data")

    @property
    def cache_path(self) -> Path:
        safe_symbol = self.symbol.lower().replace("/", "_").replace(":", "_")
        return self.cache_dir / f"funding_{self.exchange.lower()}_{safe_symbol}.csv"


@dataclass(frozen=True)
class ExecutionCostConfig:
    spot_fee: float
    perp_fee: float
    slippage_per_leg: float = 0.0
    note: str = ""

    @property
    def round_trip_cost(self) -> float:
        # Entry and exit each trade both legs: spot + perp.
        return 2 * (self.spot_fee + self.perp_fee) + 4 * self.slippage_per_leg


# Choose the market here.
# Supported exchange values in this notebook: "kraken", "binance_usdm", "hyperliquid".
# Examples:
#   MarketConfig("kraken", "PF_XBTUSD")
#   MarketConfig("kraken", "PF_ETHUSD")
#   MarketConfig("binance_usdm", "BTCUSDT")
#   MarketConfig("binance_usdm", "ETHUSDT")
#   MarketConfig("hyperliquid", "BTC")
#   MarketConfig("hyperliquid", "ETH")
#   MarketConfig("hyperliquid", "SOL")
MARKET = MarketConfig(exchange="kraken", symbol="PF_XBTUSD")

# Execution-cost assumptions used to convert funding into a break-even threshold.
# These are base-tier defaults, not your account-specific VIP tier. Override COSTS
# directly if your actual maker/taker tier, rebates, or slippage are different.
EXECUTION_STYLE = "taker"  # "taker" is conservative; use "maker" for passive fills.
SLIPPAGE_PER_LEG = 0.0     # Fraction of notional per executed leg, e.g. 0.0001 = 1 bp.


def default_cost_config(market: MarketConfig, execution_style: str = EXECUTION_STYLE) -> ExecutionCostConfig:
    fee_table = {
        # Kraken Pro base spot tier plus Kraken Futures base tier.
        "kraken": {
            "maker": ExecutionCostConfig(spot_fee=0.0025, perp_fee=0.0002, note="Kraken Pro/Futures base maker fees"),
            "taker": ExecutionCostConfig(spot_fee=0.0040, perp_fee=0.0005, note="Kraken Pro/Futures base taker fees"),
        },
        # Binance spot base tier plus USD-M futures base tier.
        "binance_usdm": {
            "maker": ExecutionCostConfig(spot_fee=0.0010, perp_fee=0.0002, note="Binance spot/USD-M base maker fees"),
            "taker": ExecutionCostConfig(spot_fee=0.0010, perp_fee=0.0005, note="Binance spot/USD-M base taker fees"),
        },
        # Hyperliquid spot base tier plus perp base tier.
        "hyperliquid": {
            "maker": ExecutionCostConfig(spot_fee=0.0004, perp_fee=0.00015, note="Hyperliquid base maker fees"),
            "taker": ExecutionCostConfig(spot_fee=0.0007, perp_fee=0.00045, note="Hyperliquid base taker fees"),
        },
    }

    exchange = market.exchange.lower()
    style = execution_style.lower()
    if exchange not in fee_table or style not in fee_table[exchange]:
        raise ValueError(f"No default fee config for {exchange=} and {style=}.")

    base = fee_table[exchange][style]
    return ExecutionCostConfig(
        spot_fee=base.spot_fee,
        perp_fee=base.perp_fee,
        slippage_per_leg=SLIPPAGE_PER_LEG,
        note=base.note,
    )


COSTS = default_cost_config(MARKET)
ROUND_TRIP_COST = COSTS.round_trip_cost

TRAIN_FRAC = 0.40
HOLD_HOURS_FOR_BREAKEVEN = 168  # one-week yardstick
ENTRY_MULT = 2.0
EXIT_MULT = 0.5

# Baseline filter beliefs from the original script
Q_BELIEF = 1e-13
R_BELIEF = 7e-11

# Set True to study the notebook without downloading exchange data.
USE_SYNTHETIC_DATA = False

print(f"Configuration loaded for {MARKET.exchange}:{MARKET.symbol}.")
print(f"Cost model: {EXECUTION_STYLE} fills, round-trip cost = {ROUND_TRIP_COST:.4%} ({COSTS.note})")

: 

## 1. Load the funding-rate observations

The observed variable is $z_t$: the relative perpetual funding rate observed at time $t$ for the selected market.

The relative funding rate is expressed as a fraction of notional per funding interval. For example, $0.00001$ represents one basis point of notional.

The cell below first looks for a market-specific cached CSV. When the CSV is absent, it requests historical funding rates from the configured exchange and caches them locally. A synthetic fallback is provided only so that the filter mechanics can be explored without an internet connection.

In [ ]:
def make_synthetic_funding(n: int = 2400, seed: int = 7) -> pd.DataFrame:
    """Create illustrative noisy funding regimes when real data are unavailable.

    The high-funding regimes are deliberately above the cost-aware entry
    threshold so that entries, exits, transaction costs, and P&L are visible
    when studying the notebook offline.
    """
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    high_regime = (t % 480) < 240
    target_level = np.where(high_regime, 4.8e-5, 2.0e-6)

    latent = np.zeros(n)
    latent[0] = target_level[0]
    for i in range(1, n):
        drift_toward_regime = 0.04 * (target_level[i] - latent[i - 1])
        latent[i] = latent[i - 1] + drift_toward_regime + rng.normal(0, 6e-7)

    observations = latent + rng.normal(0, 8e-6, n)
    idx = pd.date_range("2025-01-01", periods=n, freq="h", tz="UTC")
    return pd.DataFrame({"funding": observations, "latent_for_demo_only": latent}, index=idx)


def fetch_kraken_funding(symbol: str) -> pd.DataFrame:
    import requests

    url = "https://futures.kraken.com/derivatives/api/v4/historical-funding-rates"
    response = requests.get(url, params={"symbol": symbol}, timeout=30)
    response.raise_for_status()

    raw = pd.DataFrame(response.json()["rates"])
    if raw.empty:
        raise ValueError(f"Kraken returned no funding rows for {symbol}.")

    raw["timestamp"] = pd.to_datetime(raw["timestamp"], utc=True)
    raw = raw.set_index("timestamp").sort_index()
    return raw[["relativeFundingRate"]].rename(columns={"relativeFundingRate": "funding"})


def normalize_binance_funding_frame(raw: pd.DataFrame, symbol: str) -> pd.DataFrame:
    if raw.empty:
        raise ValueError(f"Binance USD-M returned no funding rows for {symbol}.")

    timestamp_candidates = ["fundingTime", "calc_time", "time", "timestamp"]
    funding_candidates = ["fundingRate", "last_funding_rate", "funding_rate", "funding"]

    timestamp_col = next((col for col in timestamp_candidates if col in raw.columns), None)
    funding_col = next((col for col in funding_candidates if col in raw.columns), None)
    if timestamp_col is None or funding_col is None:
        raise KeyError(f"Could not identify timestamp/funding columns in Binance data: {list(raw.columns)}")

    out = raw[[timestamp_col, funding_col]].copy()
    if pd.api.types.is_numeric_dtype(out[timestamp_col]):
        out["timestamp"] = pd.to_datetime(out[timestamp_col].astype("int64"), unit="ms", utc=True)
    else:
        out["timestamp"] = pd.to_datetime(out[timestamp_col], utc=True)
    out["funding"] = out[funding_col].astype(float)
    return out.set_index("timestamp").sort_index()[["funding"]]


def fetch_binance_usdm_funding_from_archive(symbol: str, start_time: str = "2019-09-01") -> pd.DataFrame:
    """Download Binance USD-M funding from public monthly ZIP archives.

    This avoids the geo-restricted fapi.binance.com endpoint that can return HTTP 451.
    """
    import io
    import zipfile
    import requests

    symbol = symbol.upper()
    base_url = "https://data.binance.vision/data/futures/um/monthly/fundingRate"
    start_period = pd.Period(pd.Timestamp(start_time), freq="M")
    end_period = pd.Period(pd.Timestamp.utcnow(), freq="M")
    frames = []

    for period in pd.period_range(start_period, end_period, freq="M"):
        archive_url = f"{base_url}/{symbol}/{symbol}-fundingRate-{period}.zip"
        response = requests.get(archive_url, timeout=30)
        if response.status_code == 404:
            continue
        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            csv_names = [name for name in zf.namelist() if name.endswith(".csv")]
            if not csv_names:
                continue
            with zf.open(csv_names[0]) as csv_file:
                frames.append(pd.read_csv(csv_file))

    if not frames:
        raise ValueError(f"No Binance archive funding files found for {symbol}.")

    return normalize_binance_funding_frame(pd.concat(frames, ignore_index=True), symbol)


def fetch_binance_usdm_funding(symbol: str, start_time: str | None = None) -> pd.DataFrame:
    """Download Binance USD-M perpetual funding history with simple pagination."""
    import requests

    url = "https://fapi.binance.com/fapi/v1/fundingRate"
    start_ms = 0 if start_time is None else int(pd.Timestamp(start_time, tz="UTC").timestamp() * 1000)
    rows = []

    while True:
        params = {"symbol": symbol.upper(), "startTime": start_ms, "limit": 1000}
        response = requests.get(f"{url}?{urlencode(params)}", timeout=30)
        try:
            response.raise_for_status()
        except requests.HTTPError:
            if response.status_code == 451:
                print("Binance API returned HTTP 451; using public data archive fallback.")
                return fetch_binance_usdm_funding_from_archive(symbol)
            raise

        batch = response.json()
        if not batch:
            break

        rows.extend(batch)
        last_time = int(batch[-1]["fundingTime"])
        next_start = last_time + 1
        if next_start <= start_ms:
            break
        start_ms = next_start

        if len(batch) < 1000:
            break

    return normalize_binance_funding_frame(pd.DataFrame(rows), symbol)


def fetch_hyperliquid_funding(symbol: str, start_time: str = "2023-01-01") -> pd.DataFrame:
    """Download Hyperliquid perpetual funding history for a coin such as BTC or ETH."""
    import requests

    url = "https://api.hyperliquid.xyz/info"
    coin = symbol.upper().replace("-PERP", "").replace("/USD", "").replace("USDC", "")
    start_ms = int(pd.Timestamp(start_time, tz="UTC").timestamp() * 1000)
    end_ms = int(pd.Timestamp.utcnow().timestamp() * 1000)
    chunk_ms = 500 * 60 * 60 * 1000  # Hyperliquid funding is hourly; keep requests bounded.
    rows = []

    while start_ms < end_ms:
        request_end_ms = min(start_ms + chunk_ms, end_ms)
        payload = {
            "type": "fundingHistory",
            "coin": coin,
            "startTime": start_ms,
            "endTime": request_end_ms,
        }
        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        batch = response.json()
        if batch:
            rows.extend(batch)

        start_ms = request_end_ms + 1

    raw = pd.DataFrame(rows)
    if raw.empty:
        raise ValueError(f"Hyperliquid returned no funding rows for {coin}.")

    raw["timestamp"] = pd.to_datetime(raw["time"].astype("int64"), unit="ms", utc=True)
    raw["funding"] = raw["fundingRate"].astype(float)
    return raw.set_index("timestamp").sort_index()[["funding"]]


def load_funding(market: MarketConfig = MARKET, synthetic: bool = False) -> pd.DataFrame:
    if synthetic:
        print("Using synthetic funding observations for learning purposes.")
        return make_synthetic_funding()

    if market.cache_path.exists():
        out = pd.read_csv(market.cache_path, index_col=0, parse_dates=True)
        print(f"Loaded {len(out):,} cached rows from {market.cache_path}.")
    else:
        market.cache_path.parent.mkdir(parents=True, exist_ok=True)
        exchange = market.exchange.lower()
        if exchange == "kraken":
            out = fetch_kraken_funding(market.symbol)
        elif exchange == "binance_usdm":
            out = fetch_binance_usdm_funding(market.symbol)
        elif exchange == "hyperliquid":
            out = fetch_hyperliquid_funding(market.symbol)
        else:
            raise ValueError("Unsupported exchange. Use 'kraken', 'binance_usdm', or 'hyperliquid'.")

        out.to_csv(market.cache_path)
        print(f"Downloaded and cached {len(out):,} {market.exchange} rows to {market.cache_path}.")

    if "funding" not in out.columns:
        raise KeyError("Expected a normalized 'funding' column.")

    return out[["funding"]].dropna().sort_index()


try:
    funding_df = load_funding(synthetic=USE_SYNTHETIC_DATA)
except Exception as exc:
    print(f"Real-data load failed: {type(exc).__name__}: {exc}")
    print("Falling back to synthetic data. Set USE_SYNTHETIC_DATA=True intentionally for repeated offline use.")
    funding_df = load_funding(synthetic=True)

funding_df.head()

In [ ]:
print(f"Market: {MARKET.exchange}:{MARKET.symbol}")
print(f"Cache file: {MARKET.cache_path}")
print(f"Date range: {funding_df.index.min()} to {funding_df.index.max()}")

In [ ]:
# Change MARKET in the configuration cell, then rerun from the data-loading cell downward.

In [ ]:
# Supported examples:
# MARKET = MarketConfig("kraken", "PF_XBTUSD")
# MARKET = MarketConfig("kraken", "PF_ETHUSD")
# MARKET = MarketConfig("binance_usdm", "BTCUSDT")
# MARKET = MarketConfig("binance_usdm", "ETHUSDT")
# MARKET = MarketConfig("hyperliquid", "BTC")
# MARKET = MarketConfig("hyperliquid", "ETH")
# MARKET = MarketConfig("hyperliquid", "SOL")

In [ ]:
# Raw data inspection
funding = funding_df["funding"].to_numpy(dtype=float)


def infer_bar_hours(index: pd.DatetimeIndex, default_hours: float = 1.0) -> float:
    if len(index) < 2:
        return default_hours
    diffs = index.to_series().diff().dropna().dt.total_seconds() / 3600
    return float(diffs.median()) if not diffs.empty else default_hours


FUNDING_INTERVAL_HOURS = infer_bar_hours(funding_df.index)
BARS_PER_YEAR = 24 * 365 / FUNDING_INTERVAL_HOURS

raw_summary = pd.Series({
    "exchange": MARKET.exchange,
    "symbol": MARKET.symbol,
    "bars": len(funding),
    "median hours per funding bar": FUNDING_INTERVAL_HOURS,
    "bars per year used for annualisation": BARS_PER_YEAR,
    "mean funding per bar": funding.mean(),
    "standard deviation per bar": funding.std(),
    "minimum": funding.min(),
    "maximum": funding.max(),
    "fraction positive": (funding > 0).mean(),
})
display(raw_summary.to_frame("value"))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(funding_df.index, funding_df["funding"], linewidth=0.65)
ax.axhline(0, linewidth=0.8)
ax.set_title(f"Observed funding rate: {MARKET.exchange}:{MARKET.symbol}")
ax.set_ylabel("relative funding rate per funding bar")
ax.grid(alpha=0.3)
plt.show()

## 2. The state-space model

We assume there is an unobserved underlying funding level $x_t$, while the selected exchange gives us a noisy observation $z_t$.

**State evolution:**

$$
x_t = x_{t-1} + w_t, \qquad w_t \sim \mathcal{N}(0, Q)
$$

The underlying funding level is assumed to drift as a random walk. $Q$ determines how quickly it may change.

**Measurement equation:**

$$
z_t = x_t + v_t, \qquad v_t \sim \mathcal{N}(0, R)
$$

$R$ determines how noisy we think an individual observed print is.

Interpretation:

| Parameter | Small value | Large value |
|---|---|---|
| $Q$ | True funding changes slowly; smoother estimate | True funding can move quickly; responsive estimate |
| $R$ | Trust each new observation | Distrust each observation; stronger smoothing |

In this scalar random-walk filter, the smoothing behaviour is driven mainly by the ratio $Q/R$.

## 3. One Kalman step, exposed line by line

At time $t$, the filter first predicts:

$$
x_{\mathrm{pred},t} = \hat{x}_{t-1}
$$

$$
P_{\mathrm{pred},t} = P_{t-1} + Q
$$

Then it observes $z_t$ and updates:

$$
\nu_t = z_t - x_{\mathrm{pred},t}
$$

$$
K_t = \frac{P_{\mathrm{pred},t}}{P_{\mathrm{pred},t} + R}
$$

$$
\hat{x}_t = x_{\mathrm{pred},t} + K_t\nu_t
$$

$$
P_t = (1-K_t)P_{\mathrm{pred},t}
$$

The Kalman gain $K_t$ controls how much the estimate moves toward the latest observation.

In [ ]:
def kalman_step_detailed(x_previous: float, P_previous: float, z: float, Q: float, R: float) -> dict:
    """Perform one scalar Kalman predict/update step and return every intermediate value."""
    x_pred = x_previous
    P_pred = P_previous + Q

    innovation = z - x_pred
    K = P_pred / (P_pred + R)

    x_hat = x_pred + K * innovation
    P_hat = (1.0 - K) * P_pred

    return {
        "x_previous": x_previous,
        "P_previous": P_previous,
        "observation_z": z,
        "Q": Q,
        "R": R,
        "x_pred": x_pred,
        "P_pred": P_pred,
        "innovation": innovation,
        "kalman_gain_K": K,
        "x_hat": x_hat,
        "P_hat": P_hat,
    }


# Use an early observation to inspect the arithmetic of one step.
example_x = funding[0]
example_P = R_BELIEF
example_z = funding[1]

one_step = kalman_step_detailed(example_x, example_P, example_z, Q_BELIEF, R_BELIEF)
display(pd.Series(one_step).to_frame("value"))

print(f"The estimate moved {one_step['kalman_gain_K']:.2%} of the way from the prediction toward the new observation.")

## 4. Run the filter through the whole series

The function below returns more than the final smoothed series. It records every intermediate quantity so that the mechanics can be graphed and inspected bar by bar.

In [ ]:
def filter_with_diagnostics(z: np.ndarray, Q: float, R: float, initial_P: float = 1.0) -> pd.DataFrame:
    """Run the scalar filter and record estimate, variance, gain, and innovation at every observation."""
    z = np.asarray(z, dtype=float)
    if z.size == 0:
        raise ValueError("The observation series is empty.")

    records = []
    x = z[0]
    P = initial_P

    for t, observation in enumerate(z):
        result = kalman_step_detailed(x, P, observation, Q, R)
        records.append(result)
        x = result["x_hat"]
        P = result["P_hat"]

    return pd.DataFrame(records)


diag = filter_with_diagnostics(funding, Q_BELIEF, R_BELIEF)
diag.index = funding_df.index
diag[["observation_z", "x_hat", "innovation", "kalman_gain_K", "P_hat"]].head(10)

: 

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(funding_df.index, funding, linewidth=0.6, alpha=0.55, label="raw funding observation z")
ax.plot(diag.index, diag["x_hat"], linewidth=1.35, label="filtered estimate x_hat")
ax.axhline(0, linewidth=0.7)
ax.set_title(f"Raw observations vs. Kalman estimate (Q/R = {Q_BELIEF / R_BELIEF:.2e})")
ax.set_ylabel("relative funding rate")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(diag.index, diag["kalman_gain_K"], linewidth=1.0)
ax.set_title("Kalman gain through time")
ax.set_ylabel("K")
ax.grid(alpha=0.3)
plt.show()

display(diag[["observation_z", "x_hat", "innovation", "kalman_gain_K", "P_pred", "P_hat"]].tail())

## 5. See the effect of changing $Q/R$

Rather than treating $Q$ and $R$ as abstract values, compare several smoothing choices directly:

- Low $Q/R$: the hidden funding level is assumed to move slowly.
- High $Q/R$: the hidden funding level is allowed to respond quickly to new funding prints.

The original strategy searches over values of $Q/R$ on the training set.

In [ ]:
q_over_r_values = [1e-4, 1e-3, 1e-2, 1e-1]

comparison = pd.DataFrame({"raw funding": funding}, index=funding_df.index)
gain_summary = []

for ratio in q_over_r_values:
    Q = ratio * R_BELIEF
    d = filter_with_diagnostics(funding, Q, R_BELIEF)
    comparison[f"filtered Q/R={ratio:g}"] = d["x_hat"].to_numpy()
    gain_summary.append({
        "Q/R": ratio,
        "ending Kalman gain": d["kalman_gain_K"].iloc[-1],
        "std of filtered estimate": d["x_hat"].std(),
    })

# Plot a bounded window so the smoothing differences remain visible.
window = min(500, len(comparison))
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(comparison.index[:window], comparison["raw funding"].iloc[:window], linewidth=0.55, alpha=0.45, label="raw")
for ratio in q_over_r_values:
    ax.plot(comparison.index[:window], comparison[f"filtered Q/R={ratio:g}"].iloc[:window], linewidth=1.2, label=f"Q/R={ratio:g}")
ax.set_title(f"How Q/R changes smoothing: first {window} bars")
ax.set_ylabel("relative funding rate")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

display(pd.DataFrame(gain_summary).set_index("Q/R"))

## 6. Costs and trading thresholds

A positive filtered funding rate is insufficient on its own. Opening and closing both legs costs money.

Let $\theta_{\mathrm{BE}}$ be the simplified per-funding-bar cost break-even yardstick:

$$
\theta_{\mathrm{BE}} = \frac{c_{\mathrm{roundtrip}}}{\mathrm{hold\ bars}}
$$

The notebook infers the median hours per funding bar from the selected market, then converts the holding-period yardstick from hours into bars. The rule applies two thresholds:

$$
\theta_{\mathrm{entry}} = 2.0\,\theta_{\mathrm{BE}}
$$

$$
\theta_{\mathrm{exit}} = 0.5\,\theta_{\mathrm{BE}}
$$

Using a higher entry threshold and lower exit threshold creates **hysteresis**. It prevents frequent entry/exit changes when the estimated funding level hovers close to the cost boundary.

In [ ]:
hold_bars_for_breakeven = HOLD_HOURS_FOR_BREAKEVEN / FUNDING_INTERVAL_HOURS
breakeven_per_bar = ROUND_TRIP_COST / hold_bars_for_breakeven
entry_threshold = ENTRY_MULT * breakeven_per_bar
exit_threshold = EXIT_MULT * breakeven_per_bar

threshold_table = pd.Series({
    "execution style": EXECUTION_STYLE,
    "cost model note": COSTS.note,
    "spot fee per fill": COSTS.spot_fee,
    "perp fee per fill": COSTS.perp_fee,
    "slippage per leg": COSTS.slippage_per_leg,
    "round-trip cost": ROUND_TRIP_COST,
    "holding-period yardstick in hours": HOLD_HOURS_FOR_BREAKEVEN,
    "median hours per funding bar": FUNDING_INTERVAL_HOURS,
    "holding-period yardstick in bars": hold_bars_for_breakeven,
    "break-even per funding bar": breakeven_per_bar,
    "entry threshold per funding bar": entry_threshold,
    "exit threshold per funding bar": exit_threshold,
})
display(threshold_table.to_frame("value"))

print(f"Raw bars above break-even: {(funding > breakeven_per_bar).mean():.2%}")

## 7. Position rule and backtest alignment

Position convention:

- `pos = 1`: long spot and short perpetual.
- `pos = 0`: flat.

The important time alignment is:

$$
\mathrm{position}_t \longrightarrow \mathrm{funding}_{t+1}
$$

This is why the P&L code uses `pos[:-1] * funding[1:]`. Using a decision made after seeing $z_t$ to earn $z_t$ would make the backtest look ahead.

In [ ]:
def make_positions(x_hat: np.ndarray, entry_thr: float, exit_thr: float) -> np.ndarray:
    """Turn the filtered estimate into a 0/1 carry position with hysteresis."""
    x_hat = np.asarray(x_hat, dtype=float)
    pos = np.zeros(len(x_hat))

    for t in range(1, len(x_hat)):
        if pos[t - 1] == 0:
            pos[t] = 1.0 if x_hat[t] > entry_thr else 0.0
        else:
            pos[t] = 0.0 if x_hat[t] < exit_thr else 1.0

    return pos


def backtest(
    funding: np.ndarray,
    x_hat: np.ndarray,
    entry_thr: float,
    exit_thr: float,
    round_trip_cost: float = ROUND_TRIP_COST,
    index=None,
) -> pd.DataFrame:
    """Calculate positions, correctly lagged funding P&L, costs, and equity."""
    pos = make_positions(x_hat, entry_thr, exit_thr)

    pnl_gross = np.zeros(len(funding))
    pnl_gross[1:] = pos[:-1] * funding[1:]  # no look-ahead: yesterday's decision earns today's funding

    turnover = np.abs(np.diff(pos, prepend=0.0))
    costs = (round_trip_cost / 2.0) * turnover
    pnl = pnl_gross - costs

    return pd.DataFrame({
        "funding": funding,
        "x_hat": x_hat,
        "pos": pos,
        "turnover": turnover,
        "pnl_gross": pnl_gross,
        "costs": costs,
        "pnl": pnl,
        "equity": np.cumsum(pnl),
    }, index=index)


baseline_bt = backtest(
    funding=funding,
    x_hat=diag["x_hat"].to_numpy(),
    entry_thr=entry_threshold,
    exit_thr=exit_threshold,
    index=funding_df.index,
)

baseline_bt.head(12)

In [ ]:
# A small table designed to make the time lag visible.
event_rows = baseline_bt.index[(baseline_bt["turnover"] > 0)].tolist()
if event_rows:
    first_change_loc = baseline_bt.index.get_loc(event_rows[0])
    start = max(0, first_change_loc - 2)
    stop = min(len(baseline_bt), first_change_loc + 4)
    display(baseline_bt.iloc[start:stop][["funding", "x_hat", "pos", "turnover", "pnl_gross", "costs", "pnl"]])
else:
    print("There were no position changes under the baseline thresholds and filter parameters.")

## 8. Performance report function

Sharpe ratio here is calculated from per-bar P&L and annualised using the inferred number of funding bars per year. Let $r_t$ be per-bar P&L as a fraction of notional:

$$
\mathrm{Sharpe} =
\frac{\operatorname{mean}(r_t)}{\operatorname{std}(r_t)}
\sqrt{\mathrm{bars\ per\ year}}
$$

This simple implementation does not subtract a risk-free rate.

In [ ]:
def performance_report(bt: pd.DataFrame, label: str, bars_per_year: float = BARS_PER_YEAR) -> pd.Series:
    pnl = bt["pnl"].to_numpy()
    equity = bt["equity"].to_numpy()
    pos = bt["pos"].to_numpy()

    entries = int((np.diff(pos) > 0).sum())
    sd = pnl.std()
    sharpe = pnl.mean() / sd * np.sqrt(bars_per_year) if sd > 0 else np.nan
    running_peak = np.maximum.accumulate(equity) if len(equity) else np.array([])
    max_drawdown = (equity - running_peak).min() if len(equity) else 0.0

    result = pd.Series({
        "bars": len(bt),
        "entries": entries,
        "time in position": pos.mean() if len(pos) else np.nan,
        "gross P&L (fraction of notional)": bt["pnl_gross"].sum(),
        "cost drag": bt["costs"].sum(),
        "net P&L (fraction of notional)": equity[-1] if len(equity) else 0.0,
        "annualised Sharpe": sharpe,
        "max drawdown": max_drawdown,
    }, name=label)
    return result


display(performance_report(baseline_bt, "Baseline parameters").to_frame())

## 9. Train/test split

Parameters must not be chosen after seeing the period used for headline evaluation.

- **Training slice:** select the smoothing level $Q/R$.
- **Test slice:** evaluate the frozen decision rule on unseen historical data.

The filter remains causal: at each bar it uses only current and previous observed funding prints. Running it through the full chronological sequence is valid because no future observation enters an earlier estimate.

In [ ]:
split_idx = int(len(funding) * TRAIN_FRAC)
train_funding = funding[:split_idx]
test_funding = funding[split_idx:]

print(f"Training bars: {len(train_funding):,} ({TRAIN_FRAC:.0%})")
print(f"Test bars:     {len(test_funding):,} ({1 - TRAIN_FRAC:.0%})")
print(f"Split timestamp: {funding_df.index[split_idx] if split_idx < len(funding_df) else 'n/a'}")

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(funding_df.index, funding, linewidth=0.6)
ax.axvline(funding_df.index[split_idx], linestyle="--", linewidth=1.2, label="train/test split")
ax.set_title("Chronological train/test split")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 10. Tune $Q/R$ on training data only

The original program does not statistically estimate the true $Q$ and $R$. Instead, it holds $R$ fixed, tries several $Q/R$ ratios, and chooses the smoothing behaviour that produced the best training Sharpe after costs.

This is a strategy-parameter search. It is kept separate from the held-out test period.

In [ ]:
def tune_q_over_r_on_train(
    funding_train: np.ndarray,
    R: float,
    entry_thr: float,
    exit_thr: float,
    candidate_ratios=None,
) -> tuple[float, float, pd.DataFrame]:
    if candidate_ratios is None:
        candidate_ratios = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]

    results = []
    for ratio in candidate_ratios:
        Q = ratio * R
        d = filter_with_diagnostics(funding_train, Q, R)
        bt = backtest(funding_train, d["x_hat"].to_numpy(), entry_thr, exit_thr)
        metrics = performance_report(bt, f"Q/R={ratio:g}")
        results.append({
            "Q/R": ratio,
            "Q": Q,
            "R": R,
            "annualised Sharpe": metrics["annualised Sharpe"],
            "net P&L": metrics["net P&L (fraction of notional)"],
            "entries": metrics["entries"],
            "time in position": metrics["time in position"],
        })

    search_results = pd.DataFrame(results).sort_values("annualised Sharpe", ascending=False).reset_index(drop=True)
    best_ratio = float(search_results.loc[0, "Q/R"])
    best_Q = best_ratio * R
    return best_Q, R, search_results


best_Q, best_R, tuning_results = tune_q_over_r_on_train(
    train_funding,
    R=R_BELIEF,
    entry_thr=entry_threshold,
    exit_thr=exit_threshold,
)

display(tuning_results)
print(f"Selected on training data only: Q/R = {best_Q / best_R:g}")

## 11. Freeze the selected parameters and evaluate the held-out period

The selected $Q$, $R$, entry threshold, and exit threshold are now fixed. The test result is the relevant out-of-sample result in this notebook.

In [ ]:
# Run the selected causal filter through the chronological series.
selected_diag = filter_with_diagnostics(funding, best_Q, best_R)

full_bt = backtest(
    funding,
    selected_diag["x_hat"].to_numpy(),
    entry_threshold,
    exit_threshold,
    index=funding_df.index,
)

train_bt = full_bt.iloc[:split_idx].copy()
train_bt["equity"] = train_bt["pnl"].cumsum()

test_bt = full_bt.iloc[split_idx:].copy()
test_bt["equity"] = test_bt["pnl"].cumsum()

reports = pd.concat([
    performance_report(train_bt, "TRAIN: tuned on this period"),
    performance_report(test_bt, "TEST: held out evaluation"),
], axis=1)

display(reports)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(full_bt.index, full_bt["funding"], linewidth=0.55, alpha=0.45, label="raw funding")
ax.plot(full_bt.index, full_bt["x_hat"], linewidth=1.2, label="selected Kalman estimate")
ax.axhline(entry_threshold, linestyle="--", linewidth=0.9, label="entry threshold")
ax.axhline(exit_threshold, linestyle=":", linewidth=0.9, label="exit threshold")
ax.axhline(breakeven_per_bar, linestyle="-.", linewidth=0.9, label="break-even/bar")
ax.axvline(full_bt.index[split_idx], linewidth=1.1, label="train/test split")
ax.set_title(f"Filtered funding signal and trading thresholds (selected Q/R = {best_Q / best_R:g})")
ax.set_ylabel("relative funding rate per funding bar")
ax.legend(ncol=3)
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(13, 2.8))
ax.fill_between(full_bt.index, 0, full_bt["pos"], step="pre", alpha=0.5)
ax.axvline(full_bt.index[split_idx], linewidth=1.1)
ax.set_ylim(-0.05, 1.05)
ax.set_title("Position: 1 = long spot / short perpetual carry; 0 = flat")
ax.grid(alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(full_bt.index, full_bt["pnl_gross"].cumsum(), linewidth=1.0, label="gross cumulative P&L")
ax.plot(full_bt.index, full_bt["equity"], linewidth=1.3, label="net cumulative P&L")
ax.axvline(full_bt.index[split_idx], linewidth=1.1, label="train/test split")
ax.set_title("Cumulative P&L as a fraction of notional")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 12. Inspect selected trades

Each row below is an entry or exit event. The cost is charged on position changes; funding P&L is earned according to the position that was already in place before the realised funding bar.

In [ ]:
trade_events = full_bt.loc[full_bt["turnover"] > 0, [
    "funding", "x_hat", "pos", "turnover", "costs", "pnl", "equity"
]].copy()

display(trade_events.head(20))
print(f"Total position-change events: {len(trade_events)}")
print(f"Entries: {(trade_events['pos'] == 1).sum()}")
print(f"Exits:   {(trade_events['pos'] == 0).sum()}")

## 13. Parameter playground

This cell makes it easy to study a particular $Q/R$ value without changing the train/test result above. Change `ratio_to_study`, run the cell again, and inspect how trading frequency, responsiveness, and net outcome change.

In [ ]:
ratio_to_study = 1e-3   # Try: 1e-4, 1e-3, 1e-2, 1e-1
Q_to_study = ratio_to_study * R_BELIEF

study_diag = filter_with_diagnostics(funding, Q_to_study, R_BELIEF)
study_bt = backtest(
    funding,
    study_diag["x_hat"].to_numpy(),
    entry_threshold,
    exit_threshold,
    index=funding_df.index,
)

display(performance_report(study_bt, f"Study Q/R={ratio_to_study:g}").to_frame())

window = min(500, len(study_bt))
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(study_bt.index[:window], study_bt["funding"].iloc[:window], linewidth=0.55, alpha=0.45, label="raw funding")
ax.plot(study_bt.index[:window], study_bt["x_hat"].iloc[:window], linewidth=1.25, label="filtered funding")
ax.axhline(entry_threshold, linestyle="--", linewidth=0.9, label="entry threshold")
ax.axhline(exit_threshold, linestyle=":", linewidth=0.9, label="exit threshold")
ax.set_title(f"Signal behaviour under Q/R={ratio_to_study:g}: first {window} bars")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 14. What this notebook demonstrates, and what remains simplified

What is implemented:

- A scalar Kalman filter for a random-walk latent funding level.
- A causal filtered signal based only on observations available by each time.
- A long-spot / short-perpetual carry position.
- Hysteresis thresholds derived from a simple transaction-cost break-even yardstick.
- A chronological training/test split.
- Training-only selection of the $Q/R$ smoothing ratio.

Important simplifications:

- $Q/R$ is selected for historical trading performance, not estimated as the true statistical data-generating process.
- The model uses exchange-specific base fee assumptions, but your real VIP tier, maker fill rate, rebates, and slippage may differ.
- It does not explicitly model borrowing/financing costs, margin, liquidation risk, basis risk, exchange risk, or execution latency.
- The one-for-one hedge is treated as structurally delta neutral.
- A single train/test split is informative but weaker than repeated walk-forward evaluation.

Useful next research steps:

1. Replace the single split with walk-forward tuning and evaluation.
2. Replace base fee defaults with account-specific maker/taker tiers, rebates, and measured slippage.
3. Compare filtered-signal trading with raw-signal trading.
4. Estimate $Q$ and $R$ by likelihood, then compare those values with strategy-tuned values.
5. Model funding sign, settlement timing, and exchange mechanics explicitly before treating results as implementable.

In [ ]:
# Optional final comparison: raw signal versus selected Kalman signal
raw_signal_bt = backtest(
    funding,
    funding,  # use raw observation directly instead of a filtered estimate
    entry_threshold,
    exit_threshold,
    index=funding_df.index,
)

raw_test = raw_signal_bt.iloc[split_idx:].copy()
raw_test["equity"] = raw_test["pnl"].cumsum()

comparison_report = pd.concat([
    performance_report(raw_test, "TEST: raw funding signal"),
    performance_report(test_bt, "TEST: Kalman-filtered signal"),
], axis=1)

display(comparison_report)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(raw_test.index, raw_test["equity"], linewidth=1.2, label="raw-signal test equity")
ax.plot(test_bt.index, test_bt["equity"], linewidth=1.2, label="Kalman-signal test equity")
ax.set_title("Held-out test comparison: raw versus filtered signal")
ax.set_ylabel("net P&L, fraction of notional")
ax.legend()
ax.grid(alpha=0.3)
plt.show()